# T1 · Prepare — put Track 1's training material on Drive

**One job.** Write the two things Track 1 rung-1 training needs, and nothing else:

1. the EEG for a small, explicit set of recordings, and
2. the frozen DINOv2 target table, one 1536-d vector per stimulus image.

It does not epoch, does not run REVE, and does not train. Those are the next steps.

---

## The target space is a fixed specification

Track 1 scores cosine retrieval in a frozen embedding space. A head trained against a
slightly different space still fits, still reports a plausible training loss, and then
retrieves badly. There is no error message for getting this wrong, so the settings below
are copied from the competition's own config rather than chosen:

```yaml
target:
  name: HuggingFaceImage
  model_name: facebook/dinov2-giant
  layers: 0.6667
  token_aggregation: mean
  imsize: 518
brain_model_output_size: 1536
```

Source: `neuralbench/tasks/eeg/image/config.yaml`, neuralbench 0.3.1, retrieved 2026-09-24.
A copy with its provenance header is in `research/neuralbench_eeg_image_config.yaml`.

What those values resolve to, traced through `neuralset/extractors/image.py` and
`neuralset/extractors/hf.py`:

| setting | what actually happens |
|---|---|
| `layers: 0.6667` | `int(0.6667 x 40) = 26`, so **hidden state 26 of 41**. Truncation, not rounding. |
| `token_aggregation: mean` | mean over all 257 tokens, CLS included |
| `imsize: 518` | resize shorter side to 518, then the HF processor resizes to 256 and centre-crops 224 |

The `imsize` step is nearly cancelled by the processor, and neuralset warns as much. It is
kept because it is not *entirely* cancelled: resampling twice is not the same as resampling
once, and the reference table was built with it.

The embedding code in this notebook was checked against neuralset's own extractor on the
same images and agreed to `max |diff| = 0.0`. That check ran on `dinov2-base`, since the
code path is identical and dimension-agnostic but the download is 350 MB instead of 4.5 GB.

## What it costs

With the defaults below, 12 subjects contributing one training run and one test run each:

| | |
|---|---|
| EEG on Drive | ~125 MB (24 recordings) |
| embedding table | ~77 MB (~12,500 images x 1536 float32) |
| images retained | none, they are streamed and discarded |
| **Drive total** | **~200 MB** |

The embedding pass is network-bound, not GPU-bound. It is resumable: progress is written in
shards, so a dropped Colab session costs one shard, not 12,500 fetches.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'   # the folder you created

N_SUBJECTS = 12            # subjects contributing recordings
RUNS_PER_SUBJECT = 1       # training runs each, and the same number of test runs

MODEL_NAME = 'facebook/dinov2-giant'   # the scored target space; do not change to fit
DEPTH = 0.6667                         # layers, from the track config
IMSIZE = 518
EMBED_BATCH = 16           # images per forward pass
FETCH_WORKERS = 16         # concurrent downloads; the pass is network-bound
SHARD = 512                # embeddings per resumable shard

print(f'root       {ROOT}')
print(f'subjects   {N_SUBJECTS} x ({RUNS_PER_SUBJECT} train + {RUNS_PER_SUBJECT} test) runs')
print(f'target     {MODEL_NAME} @ depth {DEPTH}, mean tokens, imsize {IMSIZE}')

In [ ]:
import sys, subprocess
from pathlib import Path

SRC = Path('/content/repo/src')
if Path('../src/tracks.py').exists():
    SRC = Path('../src').resolve()
elif not SRC.exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/AGRamirezz/Neurips26-eeg-foundation-model.git',
                    '/content/repo'], check=True)
sys.path.insert(0, str(SRC))
print('helpers:', sorted(f.name for f in SRC.glob('*.py')))

In [ ]:
%pip install -q 'eegdash>=0.9.1' mne transformers torchvision pillow requests

import importlib
for mod in ('eegdash', 'mne', 'transformers', 'torchvision', 'PIL', 'requests', 'torch'):
    m = importlib.import_module(mod)
    print(f'{mod:14} {getattr(m, "__version__", "ok")}')

In [ ]:
import os

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

import tracks, acquire, miniload

ROOT = Path(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
T = tracks.get('1')
P = tracks.paths(ROOT, T)

# DINOv2-giant is 4.5 GB and is needed once. Keeping it on ephemeral /content costs a
# re-download on a new session and saves the Drive space for data that must persist.
os.environ['HF_HOME'] = '/content/hf'

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'track      {T.key} {T.name}  [{T.eegdash_id}]')
print(f'raw        {P["raw"]}')
print(f'prepared   {P["prepared"]}')
print(f'device     {DEVICE}')
if DEVICE == 'cpu':
    print('\nNo GPU. dinov2-giant on CPU will take hours for this many images.')
    print('Switch the runtime to a GPU, or set MODEL_NAME to facebook/dinov2-base for a dry run.')

## Step 1 — choose the recordings

Alljoined splits its stimuli by run. Runs 01-04 present the 200 fixed test images, runs 05-19
present training images, and the two sets do not intersect. The run number *is* the
cross-stimulus split, which is why this selects by run rather than by byte budget.

Both sides of the split come from the same session for a given subject. Track 1's shift is
cross-stimulus, so holding subject and session fixed keeps a local score about images rather
than about session drift.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import mne
mne.set_log_level('ERROR')
from eegdash import EEGDash, EEGDashDataset

records = [r for r in EEGDash().find({'dataset': T.eegdash_id})
           if r['bids_relpath'].startswith('sub-')]
print(f'catalogue: {len(records)} records, '
      f'{len({r["subject"] for r in records})} subjects')

train_recs, test_recs = miniload.select_run_split(
    records, n_subjects=N_SUBJECTS, runs_per_subject=RUNS_PER_SUBJECT,
    exclude_sessions=T.exclude_sessions)

for a, b in zip(train_recs, test_recs):
    print(f'  sub-{a["subject"]}  ses-{a["session"]}   '
          f'train run-{a["run"]}   test run-{b["run"]}')

selected = train_recs + test_recs
split_of = {r['bids_relpath']: s for recs, s in
            ((train_recs, 'train'), (test_recs, 'test')) for r in recs}
print(f'\n{len(selected)} recordings, ~{sum(miniload.estimate_mb(r) for r in selected):.0f} MB '
      f'in memory (the EDFs on disk are about half that)')
print(acquire.format_checks(acquire.verify_records(selected, T), 'selection:'))

## Step 2 — pull the EEG

In [ ]:
import time

os.environ['EEGDASH_CACHE_DIR'] = str(P['raw'])
ds = EEGDashDataset(records=selected, cache_dir=P['raw'])

t0 = time.time()
fetched, failed = [], []
for i, d in enumerate(ds.datasets, 1):
    try:
        _ = d.raw                      # touching .raw is what downloads
        fetched.append(d)
    except Exception as e:
        failed.append((i, f'{type(e).__name__}: {e}'))
        print(f'  record {i} failed: {type(e).__name__}: {str(e)[:90]}')
print(f'{len(fetched)}/{len(selected)} recordings in {time.time() - t0:.0f}s')

on_disk_mb = sum(f.stat().st_size for f in P['raw'].rglob('*') if f.is_file()) / 1e6
print(f'{on_disk_mb:.0f} MB on Drive')
if fetched:
    print(acquire.format_checks(acquire.verify_recording(fetched[0].raw, T), 'first recording:'))

## Step 3 — read the stimulus lists

EEGDash caches each recording's BIDS `_events.tsv` next to the EDF, so the image shown at
every onset can be read directly.

The key is `stim_file`, not `value`. The dataset's own `events.json` records that `value` is
zero-padded to five digits in eleven sessions and is a placeholder for oddball rows in seven
more. `stim_file` names the JPG in every row that has one.

In [ ]:
import pandas as pd

STIM_KINDS = {'train': 'stim_train', 'test': 'stim_test'}
events, images = {}, {'train': set(), 'test': set()}

for rec in selected:
    rel = rec['bids_relpath']
    split = split_of[rel]
    tsv = P['raw'] / T.eegdash_id / rel.replace('_eeg.edf', '_events.tsv')
    if not tsv.exists():
        print(f'  no events.tsv for {rel}')
        continue
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM_KINDS[split]].dropna(subset=['stim_file'])
    events[rel] = df[['onset', 'sample', 'stim_file']].reset_index(drop=True)
    images[split] |= set(df.stim_file)

n_ev = {s: sum(len(v) for k, v in events.items() if split_of[k] == s) for s in ('train', 'test')}
print(f'train   {n_ev["train"]:6,} events   {len(images["train"]):6,} unique images')
print(f'test    {n_ev["test"]:6,} events   {len(images["test"]):6,} unique images')

overlap = images['train'] & images['test']
print(f'\ntrain/test image overlap: {len(overlap)}   (must be 0 for a cross-stimulus split)')
assert not overlap, f'{len(overlap)} images appear in both splits'

all_images = sorted(images['train'] | images['test'])
print(f'to embed: {len(all_images):,} images  ->  '
      f'{miniload.targets_mb(len(all_images)):.0f} MB table')
print(f'local gallery for scoring: {len(images["test"])} images, '
      f'chance top-5 {5 / max(len(images["test"]), 1):.2%}')

## Step 4 — build the target table

The model is loaded once and the images are streamed past it. Nothing is kept but the
1536-d vector.

The self-test below is the guard against a silently wrong target space. It asserts the
layer index and the output dimension, which are the two things that would otherwise be
wrong without complaint.

In [ ]:
import io, requests, numpy as np
from PIL import Image as PILImage
from torchvision import transforms
import transformers
from transformers import AutoImageProcessor, AutoModel

STIM_URL = f'https://data.nemar.org/{T.eegdash_id}/v1.0.3/stimuli/{{}}'

_tf = transforms.Compose([transforms.Resize(IMSIZE), transforms.ToTensor()])
_proc = AutoImageProcessor.from_pretrained(MODEL_NAME, do_rescale=False)
_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
_session = requests.Session()
# The default pool holds 10 connections; with more workers than that, requests
# discards and re-opens sockets for every batch.
_session.mount('https://', requests.adapters.HTTPAdapter(
    pool_connections=FETCH_WORKERS, pool_maxsize=FETCH_WORKERS, max_retries=2))


def fetch(name):
    """Stream one stimulus. The nemar URL 302-redirects to presigned S3."""
    r = _session.get(STIM_URL.format(name), timeout=60)
    r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content)).convert('RGB')


def embed(pil_images):
    """The target space, exactly as neuralbench defines it. Returns (B, 1536) float32."""
    tens = [_tf(im).float() for im in pil_images]
    inputs = _proc(images=tens, return_tensors='pt')
    pv = inputs['pixel_values']
    if pv.isnan().any():                 # uniform images produce NaNs; neuralset zeroes them
        pv[pv.isnan()] = 0
        inputs['pixel_values'] = pv.float()
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        states = _model(**inputs, output_hidden_states=True).hidden_states
    if states is None:
        raise RuntimeError(f'{MODEL_NAME} returned no hidden states on transformers '
                           f'{transformers.__version__}; hidden-state collection regressed '
                           'for some encoders in transformers 5')
    stacked = torch.cat([s.unsqueeze(1) for s in states], dim=1)   # (B, layers, tokens, F)
    n_layers = stacked.shape[1]
    idx = int(DEPTH * (n_layers - 1))    # truncation, matching neuralset
    return stacked[:, idx].mean(dim=1).float().cpu().numpy(), idx, n_layers


def _safe_fetch(name):
    """Fetch, or None. Keeps one bad URL from ending a 12,000-image pass."""
    try:
        return fetch(name)
    except Exception:
        return None


probe = [fetch(n) for n in all_images[:4]]
vecs, layer_idx, n_layers = embed(probe)
print(f'{MODEL_NAME}: {n_layers} hidden states, taking index {layer_idx}')
print(f'embedding {vecs.shape}, norms {np.linalg.norm(vecs, axis=1).round(2)}')

if MODEL_NAME == 'facebook/dinov2-giant':
    assert (n_layers, layer_idx) == (41, 26), (n_layers, layer_idx)
    assert vecs.shape[1] == 1536, vecs.shape
    print('matches the scored target space')
else:
    print(f'DRY RUN on {MODEL_NAME}: this table is NOT the scored target space')
del probe, vecs

In [ ]:
from concurrent.futures import ThreadPoolExecutor

parts = P['prepared'] / 'embedding_parts'
parts.mkdir(parents=True, exist_ok=True)

done = set()
for f in sorted(parts.glob('part_*.npz')):
    done |= set(np.load(f)['keys'].tolist())
todo = [n for n in all_images if n not in done]
print(f'{len(done):,} already embedded, {len(todo):,} to go')

pool = ThreadPoolExecutor(FETCH_WORKERS)
buf_keys, buf_vecs, failures = [], [], []
shard_no = len(list(parts.glob('part_*.npz')))
t0 = time.time()


def flush():
    global buf_keys, buf_vecs, shard_no
    if not buf_keys:
        return
    np.savez(parts / f'part_{shard_no:05d}.npz',
             keys=np.array(buf_keys), emb=np.stack(buf_vecs).astype('float32'))
    shard_no += 1
    buf_keys, buf_vecs = [], []


for start in range(0, len(todo), EMBED_BATCH):
    names = todo[start:start + EMBED_BATCH]
    got, imgs = [], []
    for name, im in pool.map(lambda n: (n, _safe_fetch(n)), names):
        if im is None:
            failures.append(name)
        else:
            got.append(name)
            imgs.append(im)
    if imgs:
        vecs, _, _ = embed(imgs)
        buf_keys += got
        buf_vecs += list(vecs)
    if len(buf_keys) >= SHARD:
        flush()
    n_seen = start + len(names)
    if n_seen % (EMBED_BATCH * 20) == 0 or n_seen >= len(todo):
        rate = n_seen / max(time.time() - t0, 1e-9)
        print(f'  {n_seen:6,}/{len(todo):,}  {rate:5.1f} img/s  '
              f'eta {(len(todo) - n_seen) / max(rate, 1e-9) / 60:5.1f} min  '
              f'{len(failures)} failed')

flush()
print(f'\ndone in {(time.time() - t0) / 60:.1f} min, {len(failures)} failures')

In [ ]:
# Retry anything that failed, once, serially. A failure here is usually a transient
# 5xx from the redirect, not a missing image.
if failures:
    print(f'retrying {len(failures)}')
    retry_keys, retry_vecs, still = [], [], []
    for name in failures:
        im = _safe_fetch(name)
        if im is None:
            still.append(name)
            continue
        v, _, _ = embed([im])
        retry_keys.append(name); retry_vecs.append(v[0])
    if retry_keys:
        np.savez(parts / f'part_{shard_no:05d}.npz',
                 keys=np.array(retry_keys), emb=np.stack(retry_vecs).astype('float32'))
        shard_no += 1
    failures = still
    print(f'{len(failures)} still failing: {failures[:10]}')
else:
    print('nothing to retry')

## Step 5 — consolidate and record what was written

In [ ]:
import json

keys, mat = [], []
for f in sorted(parts.glob('part_*.npz')):
    z = np.load(f)
    keys += z['keys'].tolist()
    mat.append(z['emb'])

emb = np.concatenate(mat) if mat else np.zeros((0, 1536), 'float32')
order = {k: i for i, k in enumerate(keys)}          # last write wins on a duplicate
uniq = sorted(order)
emb = emb[[order[k] for k in uniq]]

table = P['prepared'] / 'image_embeddings.npz'
np.savez(table, keys=np.array(uniq), emb=emb,
                    model=MODEL_NAME, depth=DEPTH, imsize=IMSIZE,
                    token_aggregation='mean')
print(f'{table}  {emb.shape}  {table.stat().st_size / 1e6:.0f} MB')

missing = sorted(set(all_images) - set(uniq))
print(f'coverage: {len(uniq):,}/{len(all_images):,} images, {len(missing)} missing')
if missing:
    print(f'  missing: {missing[:10]}')

In [ ]:
splits = {rel: {'split': split_of[rel],
                'subject': rel.split('/')[0].removeprefix('sub-'),
                'events': len(df),
                'images': sorted(set(df.stim_file))}
          for rel, df in events.items()}

meta = {
    'target': {'model': MODEL_NAME, 'layers': DEPTH, 'token_aggregation': 'mean',
               'imsize': IMSIZE, 'dim': int(emb.shape[1]),
               'source': 'neuralbench/tasks/eeg/image/config.yaml (neuralbench 0.3.1)'},
    'epoch': {'start_s': -0.2, 'duration_s': 1.0, 'baseline_s': [0.0, 0.2],
              'note': 'from the same config; ISI is ~200 ms so epochs overlap heavily'},
    'table': {'path': str(table), 'images': len(uniq), 'missing': missing},
    'recordings': splits,
    'counts': {'train_events': n_ev['train'], 'test_events': n_ev['test'],
               'train_images': len(images['train']), 'test_images': len(images['test'])},
}
(P['prepared'] / 'prepare_manifest.json').write_text(json.dumps(meta, indent=2))
print(f'manifest -> {P["prepared"] / "prepare_manifest.json"}')

acquire.write_manifest(P['manifest'], acquire.build_manifest(
    T, selected, acquire.verify_records(selected, T), mb=on_disk_mb,
    notes=f'{N_SUBJECTS} subjects, run split, {len(uniq)} image embeddings'))

used = sum(f.stat().st_size for f in P['base'].rglob('*') if f.is_file()) / 1e6
print(f'\nDrive used by track 1: {used:.0f} MB')
print(f'  EEG        {on_disk_mb:6.0f} MB')
print(f'  embeddings {table.stat().st_size / 1e6:6.0f} MB')
print('\nNext: epoch the EEG at -0.2 to 0.8 s, run REVE over the windows, fit the ridge head.')
print('Note: these recordings carry no montage. REVE is position-aware, so the next')
print('step has to set one (the 32 channel names are standard 10-20) before encoding.')